In [5]:
from fairchem.core.datasets import AseDBDataset

dataset_path = "rattled-relax"
config_kwargs = {} # see tutorial on additional configuration

dataset = AseDBDataset(config=dict(src=dataset_path, **config_kwargs))

# atoms objects can be retrieved by index
atoms = dataset.get_atoms(0)


In [6]:
atoms

Atoms(symbols='Cr2Ge2H2', pbc=True, cell=[[3.05344565, -0.03810996, -0.00669937], [-1.61031097, 2.55030612, 0.00573872], [-0.02303385, 0.01374638, 8.3456541]], calculator=SinglePointCalculator(...))

In [8]:
import glob                                                                                                      
import numpy as np                                                                                             
import pandas as pd                                                                                              
import ase.db                                                                                                    
from multiprocessing import Pool, cpu_count
from tqdm import tqdm                                                                                            

def query_shard(shard):
  records = []
  db = ase.db.connect(shard)
  for row in db.select("Ta"):                                                                                  
      atoms = row.toatoms()
      calc = atoms.calc                                                                                        
      records.append({                                                                                       
          "shard": shard,
          "row_id": row.id,
          "formula": atoms.get_chemical_formula(),
          "n_atoms": len(atoms),                                                                               
          "energy": calc.results["energy"],
          "energy_per_atom": calc.results["energy"] / len(atoms),                                              
          "forces_max": np.abs(calc.results["forces"]).max(),                                                
          "forces_rms": np.sqrt((calc.results["forces"] ** 2).mean()),                                         
          "stress": calc.results["stress"].tolist(),
          "cell": atoms.cell.tolist(),                                                                         
      })                                                                                                     
  return records

shards = sorted(glob.glob("rattled-relax/db_*.aselmdb"))                                                         
shards = [s for s in shards if "lock" not in s]

n_workers = cpu_count()                                                                                        
print(f"Using {n_workers} workers for {len(shards)} shards")

with Pool(n_workers) as pool:
  results = list(tqdm(pool.imap(query_shard, shards), total=len(shards)))                                      

records = [r for shard_records in results for r in shard_records]                                                
df_ta = pd.DataFrame(records)
print(f"Found {len(df_ta)} Ta-containing structures")                                                            
df_ta.head() 

Using 192 workers for 500 shards


100%|██████████| 500/500 [01:08<00:00,  7.26it/s]


Found 335680 Ta-containing structures


,shard,row_id,formula,n_atoms,energy,energy_per_atom,forces_max,forces_rms,stress,cell
0,rattled-relax/db_0.aselmdb,577,Ge4Mo4Ta4,12,-109.506855,-9.125571,0.532261,0.210279,"[0.01737259195312807, 0.014442446500340296, 0....","[[5.37074275, -0.09594605, 0.15540653], [-0.11..."
1,rattled-relax/db_0.aselmdb,578,Ge4Mo4Ta4,12,-109.518279,-9.126523,0.497664,0.197403,"[0.02635992912498752, 0.026059184217251067, 0....","[[5.37131966, -0.09866979, 0.14632438], [-0.11..."
2,rattled-relax/db_0.aselmdb,579,Ge4Mo4Ta4,12,-109.605144,-9.133762,0.423688,0.194434,"[0.010942054541556322, 0.014207259882892433, 0...","[[5.33751659, -0.09678138, 0.12615595], [-0.11..."
3,rattled-relax/db_0.aselmdb,580,Ge4Mo4Ta4,12,-109.659655,-9.138305,0.563245,0.219557,"[-0.005452552013671226, 0.0016510748351072182,...","[[5.29118478, -0.09419306, 0.09851233], [-0.10..."
4,rattled-relax/db_0.aselmdb,581,Ge4Mo4Ta4,12,-109.724286,-9.143690,0.542921,0.201981,"[-0.005010233800560524, -0.0006339778628730819...","[[5.28137963, -0.08053733, 0.09191482], [-0.09..."


In [9]:
import glob                                                                                                      
import numpy as np                                                                                             
import pandas as pd                                                                                              
import ase.db                                                                                                    
from multiprocessing import Pool, cpu_count
from tqdm import tqdm                                                                                            

def query_shard(shard):
  records = []
  db = ase.db.connect(shard)
  for row in db.select("W"):                                                                                  
      atoms = row.toatoms()
      calc = atoms.calc                                                                                        
      records.append({                                                                                       
          "shard": shard,
          "row_id": row.id,
          "formula": atoms.get_chemical_formula(),
          "n_atoms": len(atoms),                                                                               
          "energy": calc.results["energy"],
          "energy_per_atom": calc.results["energy"] / len(atoms),                                              
          "forces_max": np.abs(calc.results["forces"]).max(),                                                
          "forces_rms": np.sqrt((calc.results["forces"] ** 2).mean()),                                         
          "stress": calc.results["stress"].tolist(),
          "cell": atoms.cell.tolist(),                                                                         
      })                                                                                                     
  return records

shards = sorted(glob.glob("rattled-relax/db_*.aselmdb"))                                                         
shards = [s for s in shards if "lock" not in s]

n_workers = cpu_count()                                                                                        
print(f"Using {n_workers} workers for {len(shards)} shards")

with Pool(n_workers) as pool:
  results = list(tqdm(pool.imap(query_shard, shards), total=len(shards)))                                      

records = [r for shard_records in results for r in shard_records]                                                
df_ta = pd.DataFrame(records)
print(f"Found {len(df_ta)} Ta-containing structures")                                                            
df_ta.head() 

Using 192 workers for 500 shards


100%|██████████| 500/500 [01:09<00:00,  7.18it/s]


Found 325231 Ta-containing structures


,shard,row_id,formula,n_atoms,energy,energy_per_atom,forces_max,forces_rms,stress,cell
0,rattled-relax/db_0.aselmdb,362,Ge2V2W2,6,-43.076574,-7.179429,1.070118,0.357716,"[0.1555928551220063, 0.15090926359246998, 0.17...","[[5.32751804, -0.16938394, 0.12104861], [-0.10..."
1,rattled-relax/db_0.aselmdb,363,Ge2V2W2,6,-46.430753,-7.738459,0.775379,0.361413,"[0.16177576728124982, 0.16075432907727524, 0.1...","[[5.13806886, -0.12749983, 0.09111654], [-0.07..."
2,rattled-relax/db_0.aselmdb,364,Ge2V2W2,6,-48.479857,-8.079976,1.126959,0.571722,"[0.1418981014692884, 0.15505249338612745, 0.13...","[[5.00969886, -0.09911933, 0.07083468], [-0.05..."
3,rattled-relax/db_0.aselmdb,365,Ge2V2W2,6,-50.760126,-8.460021,2.565584,1.109653,"[0.028657244122732364, 0.09307935865119323, -0...","[[4.7855318, -0.04955966, 0.03541734], [-0.029..."
4,rattled-relax/db_0.aselmdb,366,Ge2V2W2,6,-50.795258,-8.465876,2.906925,1.209475,"[-0.005073682885187187, 0.07113488602966388, -...","[[4.74523358, -0.04065039, 0.02905041], [-0.02..."


In [10]:
import glob                                                                                                      
import numpy as np                                                                                             
import pandas as pd                                                                                              
import ase.db                                                                                                    
from multiprocessing import Pool, cpu_count
from tqdm import tqdm                                                                                            

def query_shard(shard):
  records = []
  db = ase.db.connect(shard)
  for row in db.select("Nb"):                                                                                  
      atoms = row.toatoms()
      calc = atoms.calc                                                                                        
      records.append({                                                                                       
          "shard": shard,
          "row_id": row.id,
          "formula": atoms.get_chemical_formula(),
          "n_atoms": len(atoms),                                                                               
          "energy": calc.results["energy"],
          "energy_per_atom": calc.results["energy"] / len(atoms),                                              
          "forces_max": np.abs(calc.results["forces"]).max(),                                                
          "forces_rms": np.sqrt((calc.results["forces"] ** 2).mean()),                                         
          "stress": calc.results["stress"].tolist(),
          "cell": atoms.cell.tolist(),                                                                         
      })                                                                                                     
  return records

shards = sorted(glob.glob("rattled-relax/db_*.aselmdb"))                                                         
shards = [s for s in shards if "lock" not in s]

n_workers = cpu_count()                                                                                        
print(f"Using {n_workers} workers for {len(shards)} shards")

with Pool(n_workers) as pool:
  results = list(tqdm(pool.imap(query_shard, shards), total=len(shards)))                                      

records = [r for shard_records in results for r in shard_records]                                                
df_ta = pd.DataFrame(records)
print(f"Found {len(df_ta)} Ta-containing structures")                                                            
df_ta.head() 

Using 192 workers for 500 shards


100%|██████████| 500/500 [01:17<00:00,  6.46it/s]


Found 245908 Ta-containing structures


,shard,row_id,formula,n_atoms,energy,energy_per_atom,forces_max,forces_rms,stress,cell
0,rattled-relax/db_0.aselmdb,2462,Cu4Ge4Nb4,12,-65.829643,-5.485804,1.837533,0.465035,"[0.128790273975517, 0.10234949964387846, 0.110...","[[6.23058585, 0.0926335, 0.02371511], [0.08973..."
1,rattled-relax/db_0.aselmdb,2463,Cu4Ge4Nb4,12,-72.653123,-6.054427,1.611722,0.546785,"[0.08930358683461324, 0.02846644096405979, 0.0...","[[5.57333476, 0.03566982, 0.00913183], [0.0345..."
2,rattled-relax/db_0.aselmdb,2464,Cu4Ge4Nb4,12,-72.748157,-6.062346,1.738074,0.604724,"[0.07270065802847596, 0.014917210455902316, 0....","[[5.53771134, 0.03258236, 0.00834141], [0.0315..."
3,rattled-relax/db_0.aselmdb,2465,Cu4Ge4Nb4,12,-72.796453,-6.066371,1.857046,0.668790,"[0.06661128079419296, 0.010449119611819517, 0....","[[5.50239859, 0.02952181, 0.00755788], [0.0285..."
4,rattled-relax/db_0.aselmdb,2466,Cu4Ge4Nb4,12,-72.089132,-6.007428,2.662853,1.221653,"[-0.028288258392741617, -0.05322227728390031, ...","[[5.33208617, 0.01476091, 0.00377894], [0.0142..."
